In [1]:
import os
import shutil
from PIL import Image
from tqdm import tqdm
import imghdr
from pathlib import Path

def copy_and_clean_data(source_dir: str, dest_dir: str):
    """
    Создаёт копию данных, удаляет битые файлы и webp, конвертирует всё в JPEG
    
    Args:
        source_dir: исходная папка с данными
        dest_dir: папка для очищенной копии
    """
    
    os.makedirs(dest_dir, exist_ok=True)
    
    stats = {
        'total_found': 0,
        'copied': 0,
        'webp_removed': 0,
        'broken_removed': 0,
        'converted': 0,
        'skipped': 0
    }
    
    print(f"Исходная папка: {source_dir}")
    print(f"Папка назначения: {dest_dir}")
    print("\n" + "="*50)
    
    for root, dirs, files in os.walk(source_dir):
        rel_path = os.path.relpath(root, source_dir)
        if rel_path == '.':
            dest_root = dest_dir
        else:
            dest_root = os.path.join(dest_dir, rel_path)
            os.makedirs(dest_root, exist_ok=True)
        
        for file in tqdm(files, desc=f"Обработка {rel_path if rel_path != '.' else 'корневая папка'}"):
            file_path = os.path.join(root, file)
            stats['total_found'] += 1
            
            file_ext = os.path.splitext(file)[1].lower()
            
            if file_ext == '.webp':
                stats['webp_removed'] += 1
                tqdm.write(f"Удалён webp: {file}")
                continue
            
            if not is_valid_image(file_path):
                stats['broken_removed'] += 1
                tqdm.write(f"Удалён битый файл: {file}")
                continue
            
            try:
                base_name = os.path.splitext(file)[0]
                new_filename = base_name + '.jpg'
                dest_path = os.path.join(dest_root, new_filename)
                
                convert_to_jpeg(file_path, dest_path)
                
                stats['copied'] += 1
                if file_ext != '.jpg':
                    stats['converted'] += 1
                    tqdm.write(f"Конвертирован: {file} -> {new_filename}")
                
            except Exception as e:
                stats['skipped'] += 1
                tqdm.write(f"Ошибка при обработке {file}: {e}")
    
    print("\n" + "="*50)
    print("СТАТИСТИКА:")
    print(f"Всего найдено файлов: {stats['total_found']}")
    print(f"Удалено webp: {stats['webp_removed']}")
    print(f"Удалено битых: {stats['broken_removed']}")
    print(f"Конвертировано в JPEG: {stats['converted']}")
    print(f"Скопировано чистых JPEG: {stats['copied']}")
    print(f"Пропущено (ошибки): {stats['skipped']}")
    print(f"\nОчищенные данные сохранены в: {dest_dir}")


def is_valid_image(file_path: str) -> bool:
    """
    Проверяет, является ли файл корректным изображением
    """
    try:
        with Image.open(file_path) as img:
            img.verify()
        return True
    except Exception:
        return False


def convert_to_jpeg(source_path: str, dest_path: str, quality: int = 95):
    """
    Конвертирует изображение в формат JPEG
    
    Args:
        source_path: путь к исходному изображению
        dest_path: путь для сохранения JPEG
        quality: качество JPEG (1-100)
    """
    with Image.open(source_path) as img:
        if img.mode in ('RGBA', 'LA', 'P'):
            rgb_img = Image.new('RGB', img.size, (255, 255, 255))
            rgb_img.paste(img, mask=img.split()[-1] if img.mode == 'RGBA' else None)
            img = rgb_img
        elif img.mode != 'RGB':
            img = img.convert('RGB')
        
        img.save(dest_path, 'JPEG', quality=quality, optimize=True)


def delete_empty_folders(directory: str):
    """
    Удаляет пустые папки после обработки
    """
    deleted = 0
    for root, dirs, files in os.walk(directory, topdown=False):
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            try:
                if not os.listdir(dir_path):
                    os.rmdir(dir_path)
                    deleted += 1
                    print(f"Удалена пустая папка: {dir_path}")
            except OSError:
                pass
    if deleted:
        print(f"Удалено пустых папок: {deleted}")


SOURCE_DIR = "source_images"
DEST_DIR = "clean_images"

copy_and_clean_data(SOURCE_DIR, DEST_DIR)

print("\nУдаление пустых папок...")
delete_empty_folders(DEST_DIR)

C:\Users\Ева\AppData\Local\Temp\ipykernel_18156\292220992.py:5: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


Исходная папка: source_images
Папка назначения: clean_images



Обработка корневая папка: 0it [00:00, ?it/s]
Обработка Peeling_paint_on_the_wall: 0it [00:00, ?it/s]
Обработка Peeling_paint_on_the_wall\10:  44%|████▍     | 4/9 [00:00<00:00, 12.66it/s]

Конвертирован: 3_similarBolhas-na-parede-1024x535.png -> 3_similarBolhas-na-parede-1024x535.jpg


Обработка Peeling_paint_on_the_wall\11:  44%|████▍     | 4/9 [00:00<00:00, 32.83it/s]

...

Конвертирован: 2_similar11.png -> 2_similar11.jpg


Обработка отслоение_краски_на_стене\9:  40%|████      | 4/10 [00:00<00:00, 39.64it/s]

Конвертирован: 3_similarbackground-3177833_1280.png -> 3_similarbackground-3177833_1280.jpg


Обработка отслоение_краски_на_стене\9: 100%|██████████| 10/10 [00:00<00:00, 19.02it/s]


СТАТИСТИКА:
Всего найдено файлов: 1281
Удалено webp: 38
Удалено битых: 172
Конвертировано в JPEG: 178
Скопировано чистых JPEG: 1071
Пропущено (ошибки): 0

Очищенные данные сохранены в: clean_images

Удаление пустых папок...
Удалена пустая папка: clean_images\Peeling_paint_on_the_wall\27
Удалена пустая папка: clean_images\Peeling_paint_on_the_wall\3
Удалена пустая папка: clean_images\Peeling_paint_on_the_wall\60
Удалена пустая папка: clean_images\Peeling_paint_on_the_wall\78
Удалена пустая папка: clean_images\отслоение_краски_на_стене\61
Удалена пустая папка: clean_images\отслоение_краски_на_стене\69
Удалено пустых папок: 6
